<a href="https://colab.research.google.com/github/aicha-bakayoko/DI-BOOTCAMP/blob/main/W8DEFI1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part 1: Environment Setup

In [ ]:
# Install packages
!pip install -q langchain langchain-community transformers sentencepiece accelerate

In [ ]:
# Check hardware
!nvidia-smi || echo "CPU runtime"

## Part 2: Load a tiny open model and build your first LLMChain

In [ ]:
# 1. Load model + tokenizer with Transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Loaded model: {model_name}")

In [ ]:
# 2. Wrap it in a Hugging Face pipeline
hf_pipeline = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50 # Limit output length to keep it quick for CPU
)

print("Hugging Face pipeline created.")

In [ ]:
# 3. Wrap the pipeline with HuggingFacePipeline and build a LLMChain using a PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Define the prompt template
prompt_template = PromptTemplate(
    template="Rewrite this text to be simpler for beginners: {text}\nRewritten text:",
    input_variables=["text"]
)

# Create the LLMChain
rewrite_chain = LLMChain(llm=llm, prompt=prompt_template, verbose=True)

print("LLMChain with PromptTemplate created.")

In [ ]:
# 4. Test with a friendly rewrite prompt
text_to_rewrite = "LangChain is a framework designed to simplify the creation of applications using large language models. It enables developers to combine LLMs with other sources of computation or knowledge."

print(f"Original text: {text_to_rewrite}\n")

rewritten_text = rewrite_chain.run(text_to_rewrite)
print(f"Rewritten text: {rewritten_text}")

## Part 3: Compose a simple two-step pipeline with LangChain Runnables

In [ ]:
from langchain.schema.runnable import RunnableSequence

# Create two prompt templates
summary_prompt = PromptTemplate(
    template="Summarize the following text:
\n{text}\n\nSummary:",
    input_variables=["text"]
)

bullet_prompt = PromptTemplate(
    template="Convert the following summary into 3 concise bullet points:\n\n{summary}\n\nBullet points:",
    input_variables=["summary"]
)

print("Summary and bullet point prompt templates created.")

In [ ]:
# Reuse the same LLM wrapper from Part 2 (llm)

# Chain them with RunnableSequence
# The output of the first chain (summary_chain) becomes the input for the second chain (bullet_chain)

summary_chain = LLMChain(llm=llm, prompt=summary_prompt)
bullet_chain = LLMChain(llm=llm, prompt=bullet_prompt)

pipeline_chain = RunnableSequence(summary_chain | bullet_chain)

print("Two-step pipeline created using RunnableSequence.")

In [ ]:
# Run on a short paragraph and inspect the bullets
long_text = (
    "The sun is the star at the center of the Solar System. It is a nearly perfect sphere of hot plasma, heated to incandescence by nuclear fusion reactions in its core. "
    "The sun radiates this energy mainly as visible light, ultraviolet light, and infrared radiation. It is by far the most important source of energy for life on Earth. "
    "The sun's diameter is about 1.39 million kilometers (864,000 miles), or 109 times that of Earth, and its mass is about 330,000 times that of Earth. "
    "It accounts for about 99.86% of the total mass of the Solar System. Roughly three-quarters of the Sun's mass consists of hydrogen (about 73% of the total mass). "
    "The rest is mostly helium (about 25% of the total mass), with much smaller quantities of heavier elements, including oxygen, carbon, neon, and iron."
)

print(f"Original text:\n{long_text}\n")

result_bullets = pipeline_chain.invoke({"text": long_text})
print("Generated Bullet Points:")
print(result_bullets)


## Part 4 (Bonus): Add a tiny conversation chain

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# Initialize ConversationBufferMemory
memory = ConversationBufferMemory()

# Create a ConversationChain using the same LLM
conversation_chain = ConversationChain(llm=llm, memory=memory, verbose=True)

print("ConversationChain with ConversationBufferMemory created.")

In [ ]:
# Send two turns (user greets, asks a follow-up)

# First turn
user_greeting = "Hi there!"
print(f"\nUser: {user_greeting}")
response1 = conversation_chain.predict(input=user_greeting)
print(f"AI: {response1}")

# Second turn (asks a follow-up, observing how memory keeps context)
user_follow_up = "Can you tell me more about LangChain?"
print(f"\nUser: {user_follow_up}")
response2 = conversation_chain.predict(input=user_follow_up)
print(f"AI: {response2}")

print("Conversation turns completed. Observe the 'Memory' in the verbose output to see context being kept.")

### Try changing the system style

To change the system style, you would typically modify the `PromptTemplate` used within the `ConversationChain`. Since `ConversationChain` uses a default prompt that incorporates memory variables, we can explicitly define a new prompt with a system message.


In [ ]:
from langchain.prompts.prompt import PromptTemplate

# Define a new prompt template with a system message
CONVERSATION_PROMPT = PromptTemplate(
    input_variables=["history", "input"],
    template=(
        "The following is a friendly conversation between a human and an AI. "
        "The AI is concise and encouraging.\n\n"
        "Current conversation:\n"
        "{history}\n"
        "Human: {input}\n"
        "AI:"
    )
)

# Create a new conversation chain with the updated prompt
styled_conversation_chain = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory(), # Use a new memory for this styled chain
    prompt=CONVERSATION_PROMPT,
    verbose=True
)

print("Styled ConversationChain created. Let's test it.")

In [ ]:
# Test the styled conversation chain

# First turn with new style
print(f"\nHuman: How can I learn more about AI?")
styled_response1 = styled_conversation_chain.predict(input="How can I learn more about AI?")
print(f"AI (styled): {styled_response1}")

# Second turn
print(f"\nHuman: What are some good resources?")
styled_response2 = styled_conversation_chain.predict(input="What are some good resources?")
print(f"AI (styled): {styled_response2}")

print("Observe the AI's responses for conciseness and encouragement based on the new system style.")